In [ ]:
# Install dependencies
%pip install anthropic python-dotenv

In [ ]:
# Load .env variables
from dotenv import load_dotenv
load_dotenv()

In [ ]:
# Create an api client
from anthropic import Anthropic

client = Anthropic()

model = "claude-sonnet-4-0"

In [ ]:
# Helper functions

def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

# Make request
def chat(messages, temperature=1.0, stop_sequences=None, system=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }

    if system:
        params["system"] = system

    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    message = client.messages.create(
        **params
    )
    return message.content[0].text


In [ ]:
# Make a starting list of messages
messages = []

# Add an initial user question of "Define quantum computing in one sentence"
add_user_message(messages, "Define quantum computing in one sentence")

# Pass a list of messages into chat
answer = chat(messages)
# answer

# Take the answer and add it as an assistant message into our list
add_assistant_message(messages, answer)
# messages

# Add in the user's follow up question
add_user_message(messages, "Write another sentence.")
# messages
# Call chat again with the list of messages to get a final aswer
answer = chat(messages)
answer

In [ ]:
# chat exercise:

messages = []

while True:
    # Get user input
    user_input = input("> ")
    print("> ", user_input)

    add_user_message(messages, user_input)
    answer = chat(messages)
    print(answer)
    add_assistant_message(messages, answer)

In [ ]:
# How to use system prompts

def chat_tutor(messages):
    system_prompt = """
    You are a patient math tutor.
    Do not directly answer a student's question.
    Guide them to a solution step by step.
    """

    message = client.messages.create(
        model=model,
        max_tokens=1000,
        messages=messages,
        system=system_prompt
    )
    return message.content[0].text

In [ ]:
system_prompt = """
    You are a patient math tutor.
    Do not directly answer a student's question.
    Guide them to a solution step by step.
"""

messages = []
add_user_message(messages, "How do I solve 5x+3=2 for x?")
answer = chat(messages, system_prompt)
answer




In [ ]:
# system prompt exercise:

system_prompt = "You are a senior python programmer. Any python code related questions answer as concise as possible. Just with code in a code markdown text that can be print and copy."

messages = []

add_user_message(messages, "Write a Python function that check string for duplicate characters.")
answer = chat(messages, system_prompt)
answer

In [ ]:
# Temperature

messages = []
add_user_message(messages, "Generate a one sentence movie idea")

answer = chat(messages, temperature=1.0)
answer

In [ ]:
# Streams

messages = []
add_user_message(messages, "Write a 1 sentence description of a fake database")

with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages
) as stream :
    for text in stream.text_stream:
        # print(text, end="")
        pass

stream.get_final_message()

In [ ]:
# Structured data

messages = []

add_user_message(messages, "Generate a very short bridge rule as json")
add_assistant_message(messages, "```json")
text = chat(messages, stop_sequences=["```"]) # stop_sequences remove ```
text

In [ ]:
import json

json.loads(text.strip())

Structure data exercise

- Use message prefilling and stop sequence only to get three different commands in a single response
- There shouldnt be any comments or explanations
- Hint: messages prefilling isnt limited to just characters like ```

In [ ]:
# Structure data exercise

messages = []

prompt = """
Generate three different sample AWS CLI commands. Each should be very short.
"""

add_user_message(messages, prompt)
add_assistant_message(messages, "here are the three commands with no comments\n ```bash")
text = chat(messages, stop_sequences=["```"])
text.strip()


In [ ]:
# print in markdwon

from IPython.display import Markdown

Markdown(text)